# Revenue, usage and temporal analysis

**Objective.** Characterise fictional recurring revenue, invoice issue amounts and product engagement across the Fieldwise observation window. Separately inspect UK retail transaction coverage and signed values using the UCI stream.

**Measurement boundaries.** Fieldwise snapshots are at episode-date grain, usage at completed episode-month grain and invoices at issued charge grain. Monthly-normalised recurring revenue describes the fictional subscription value basis. Invoice issue amounts are scheduled billings. Neither measures accounting recognition or cash collection. UCI values are signed quantity × unit price, in GBP, at retail invoice-line grain. The sources remain independent.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython import get_ipython
from IPython.display import display

# Resolve the repository before importing the local src-layout package.
project = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
        and (path / "src" / "nowa_customer_intelligence").is_dir()
    ),
    None,
)
if project is None:
    raise RuntimeError(
        "Start the Jupyter kernel from the Nowa Customer Intelligence repository "
        "root or one of its subdirectories."
    )

source_dir = str(project / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

get_ipython().run_line_magic("matplotlib", "inline")
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (10, 4), "axes.titleweight": "bold"})

## Fieldwise billing and recurring revenue

In [ ]:
from nowa_customer_intelligence.data.eda_inputs import fieldwise_reference

with fieldwise_reference() as (run, _export):
    population = run.population

billing = pd.DataFrame(
    [
        (
            charge.invoice.subscription_id,
            charge.invoice.invoice_date,
            charge.frequency.value,
            charge.plan_id,
            charge.invoice.amount,
            charge.recurring_revenue,
            charge.invoice.invoice_status.value,
        )
        for charge in population.charges
    ],
    columns=[
        "episode",
        "issued",
        "frequency",
        "plan",
        "invoice GBP",
        "recurring GBP/month",
        "status",
    ],
)
billing["month"] = pd.to_datetime(billing["issued"]).dt.to_period("M")
billing["invoice plot"] = billing["invoice GBP"].astype(float)
recurring = pd.DataFrame(
    [
        (
            row.subscription_id,
            row.snapshot_date,
            row.plan_id,
            row.billing_frequency.value,
            row.recurring_revenue,
        )
        for row in population.snapshots
    ],
    columns=["episode", "snapshot", "plan", "frequency", "recurring GBP/month"],
)
recurring["month"] = pd.to_datetime(recurring["snapshot"]).dt.to_period("M")
recurring["recurring plot"] = recurring["recurring GBP/month"].astype(float)
display(
    billing.groupby(["plan", "frequency"]).agg(
        invoices=("invoice GBP", "size"),
        billed_GBP=("invoice GBP", "sum"),
        median_invoice_GBP=("invoice GBP", "median"),
        median_monthly_normalised_GBP=("recurring GBP/month", "median"),
    )
)
display(billing.groupby("frequency").size().rename("invoices"))
display(
    billing.groupby(["frequency", "episode"]).size().groupby(level="frequency").describe().round(1)
)
display(recurring.groupby("plan")["recurring plot"].describe().round(2))
print(f"Invoice issue amount total: £{billing['invoice GBP'].sum():,.2f}")
fig, (left, right) = plt.subplots(1, 2, figsize=(14, 4), layout="constrained")
billing.groupby(["month", "frequency"])["invoice plot"].sum().unstack(fill_value=0).plot(ax=left)
left.set(
    title="Fieldwise issued invoice amounts by billing month",
    xlabel="Invoice month",
    ylabel="Issued invoice amounts (GBP)",
)
recurring.groupby(["month", "plan"])["recurring plot"].median().unstack().plot(ax=right)
right.set(
    title="Fieldwise median monthly-normalised snapshot value",
    xlabel="Snapshot month",
    ylabel="Recurring amount per episode (GBP/month)",
)
plt.show()

## Usage distribution, month coverage and billing overlap

In [ ]:
usage = pd.DataFrame(
    [
        (
            row.subscription_id,
            row.usage_month,
            row.active_users,
            row.sessions_count,
            row.events_count,
        )
        for row in population.usage
    ],
    columns=["episode", "usage month", "active users", "sessions", "events"],
)
usage["month"] = pd.to_datetime(usage["usage month"]).dt.to_period("M")
measures = ["active users", "sessions", "events"]
display(usage[measures].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(2))
coverage = (
    pd.concat(
        [
            recurring.groupby("month")["episode"].nunique().rename("snapshot episodes"),
            usage.groupby("month")["episode"].nunique().rename("usage episodes"),
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
)
display(coverage.tail(6))
invoiced = billing[["episode", "month"]].drop_duplicates().assign(invoiced=True)
usage_overlap = usage.merge(invoiced, on=["episode", "month"], how="left", validate="one_to_one")
usage_overlap["same-month invoice"] = usage_overlap["invoiced"].eq(True)
display(usage_overlap.groupby("same-month invoice")[measures].median().round(1))
display(usage_overlap["same-month invoice"].value_counts().rename("episode-month rows"))
print(
    f"Usage: {len(usage):,} rows across {usage['episode'].nunique():,} episodes, "
    f"{usage['month'].min()} to {usage['month'].max()}."
)
fig, (left, right) = plt.subplots(1, 2, figsize=(14, 4), layout="constrained")
coverage.plot(ax=left)
left.set(
    title="Fieldwise coverage by calendar month",
    xlabel="Calendar month",
    ylabel="Distinct episodes",
)
usage.groupby("month")[measures].median().plot(ax=right, logy=True)
right.set(
    title="Fieldwise median activity per observed episode-month",
    xlabel="Usage month",
    ylabel="Count (log scale)",
)
plt.show()

### Fieldwise interpretation

The reference scenario sets the catalogue and billing cadence. Annual prepayment produces larger issue amounts and usually fewer invoices, while monthly-normalised amounts divide annual value across the service basis. A snapshot records its observed state and does not establish accounting revenue. Usage is available for completed months under the generator’s coverage rule. The snapshot/usage count ratio is descriptive because their observation conventions differ. An invoice in the same month does not cause usage, and absence of an invoice does not imply absence of a live service period.

## UCI Online Retail II source audit and streaming summaries

In [ ]:
from collections import Counter, defaultdict
from decimal import Decimal

from nowa_customer_intelligence.data.public_sources.acquisition import verify_acquisition
from nowa_customer_intelligence.data.public_sources.retail import audit_retail, iter_retail_lines

source = project / "data" / "external" / "uci_online_retail_ii"
verify_acquisition(source)
workbook = source / "online_retail_II.xlsx"

# The adapter audit checks both sheets and reports repeated normalised lines.
retail_quality = audit_retail(workbook)
display(
    pd.Series(
        {key: value for key, value in retail_quality.items() if key != "sheets"},
        name="UCI source audit",
    )
)
display(pd.Series(retail_quality["sheets"], name="invoice lines"))

# Retain monthly totals and small per-customer aggregates instead of buffering the workbook.
line_counts = Counter()
signed_values = defaultdict(Decimal)
prices = Counter()
purchase_invoices = defaultdict(set)
last_purchase = {}
price_min = price_max = None
for line in iter_retail_lines(workbook):
    month = line.invoice_date.strftime("%Y-%m")
    kind = "Cancellation" if line.cancellation else "Other invoice line"
    line_counts[month, kind] += 1
    signed_values[month, kind] += line.transaction_value
    price_min = line.unit_price if price_min is None else min(price_min, line.unit_price)
    price_max = line.unit_price if price_max is None else max(price_max, line.unit_price)
    prices["Positive price"] += line.unit_price > 0
    if (
        line.customer_id is not None
        and not line.cancellation
        and line.quantity > 0
        and line.unit_price > 0
    ):
        purchase_invoices[line.customer_id].add(line.invoice_id)
        last_purchase[line.customer_id] = max(
            line.invoice_date, last_purchase.get(line.customer_id, line.invoice_date)
        )
assert sum(line_counts.values()) == retail_quality["rows"]
print(f"Observed unit price range £{price_min:,.2f} to £{price_max:,.2f}.")
print(f"Signed line value sum £{sum(signed_values.values()):,.2f}, before quality adjustments.")
print(f"Positive-price lines {prices['Positive price']:,}.")
counts_monthly = pd.Series(line_counts).unstack(fill_value=0).sort_index()
values_monthly = pd.Series(signed_values).unstack(fill_value=0).astype(float).sort_index()
anchor = retail_quality["last_invoice"].date()
known_frequency = pd.Series([len(values) for values in purchase_invoices.values()])
known_recency = pd.Series([(anchor - value.date()).days for value in last_purchase.values()])
display(
    pd.DataFrame(
        {
            "Distinct purchase invoices": known_frequency.describe(),
            "Days since last observed purchase": known_recency.describe(),
        }
    ).round(1)
)
print(f"Known customers with qualifying purchase lines: {len(purchase_invoices):,}.")
fig, (left, right) = plt.subplots(1, 2, figsize=(14, 4), layout="constrained")
counts_monthly.plot(ax=left, color=["#db8351", "#23667d"])
left.set(
    title="UCI retail invoice-line activity", xlabel="Transaction month", ylabel="Invoice lines"
)
values_monthly.plot(ax=right, color=["#db8351", "#23667d"])
right.set(
    title="UCI signed quantity × unit price",
    xlabel="Transaction month",
    ylabel="Signed value (GBP)",
)
plt.show()

## Interpretation and conclusion

The source audit retains anonymous lines, repeated lines, unusual signed quantities and prices, and missing descriptions. All contribute to the unadjusted monthly aggregates. The per-customer summaries use identified, non-cancellation lines with positive quantity and price and count distinct invoices. They describe the coverage of that subset, not the unidentified population. The final December is partial. Signed line value is sensitive to adjustments and repetition and should not be presented as accounting revenue or cash.

For later investigation, distinguish observed purchase recency from contractual churn, define a source-specific return and adjustment policy, and evaluate how missing customer IDs affect representativeness. Fieldwise engagement comparisons should use completed months and account for differences in tenure and billing frequency. Neither business association is causal, and the fictional Fieldwise distributions do not represent empirical GB behaviour.

An observed zero is a present usage measurement whose value is zero. A missing observation has no row for an expected eligible month. Structurally unavailable months, such as those outside the episode or beyond the completed-month coverage rule, should be excluded from that expectation. A row that exists in a later extract can still be unavailable at a historical prediction cutoff. Consult the recorded availability time before counting it in point-in-time coverage. Do not fill any of these three unavailable cases with observed zeros.